In [ ]:
import getpass
import pandas as pd
import psycopg2 
import json
import requests
from datetime import datetime
import numpy as np

# Variables
weekly_payout = 23.21

# Define PostgreSQL database connection parameters
# user = input("username")

api_key = getpass.getpass("Enter DataGolf API key:")
host = getpass.getpass("Enter Database Host:")
port = "5432" # The default port for PosgreSQL Server
dbname = 'postgres'
user = getpass.getpass("Enter Username:")
password = getpass.getpass("Enter Password:")

# Define a SQLAlchemy URI string for connecting to the database
# The URI structure is [DB_FLAVOR]+[DB_PYTHON_LIBRARY]://[USERNAME]:[PASSWORD]@[DB_HOST]:[PORT]/[DB_NAME]
db_URI = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"

In [ ]:
# A helper function to open a psycopg2 connection, set auto-commit to true, execute the sql, and close the connection.
# This will mainly be used for writing to the db
# def execute_sql(sql, echo=False):
#     try:
#         pg_conn = psycopg2.connect(
#             dbname=dbname,
#             user=user,
#             password=password,
#             host=host,
#             port=port
#         )
#     except psycopg2.Error as e:
#         error_message = e.pgerror
#         print("Error Connecting:", error_message)
   
#     try:
#         # Set the connection to autocommit (everything is treated as an individual transaction)
#         pg_conn.set_session(autocommit=True)
        
#         # The cursor is used to execute ddl statements.
#         pg_cursor = pg_conn.cursor() 

#         pg_cursor.execute(sql)
#         if echo:
#             print(sql)

#         results = pg_cursor.fetchall()
#         return results
    
#     except psycopg2.Error as e:
#         error_message = e.pgerror
#         print("SQL Failed:", error_message)
#         return []
    
#     finally:
#         if pg_cursor:
#             pg_cursor.close()
#         if pg_conn:
#             pg_conn.close()

In [ ]:
#function for uploading df to aws postgre SQL database
def insert_df_to_sql(table_name, df):
    try:
        pg_conn = psycopg2.connect(
            dbname=dbname,
            user=user,
            password=password,
            host=host,
            port=port
        )
    except psycopg2.Error as e:
        error_message = e.pgerror
        print("Error Connecting:", error_message)

    try:
        # Create a cursor object
        cursor = pg_conn.cursor()

        # Convert the DataFrame to a list of tuples for insertion
        insert_query = f"INSERT INTO {table_name} ({', '.join(df.columns)}) VALUES ({', '.join(['%s'] * len(df.columns))})"
        data_to_insert = df.to_records(index=False).tolist()

        # Insert data using executemany()
        cursor.executemany(insert_query, data_to_insert)

        # Commit the transaction
        pg_conn.commit()
        print("Data inserted successfully!")
    except psycopg2.Error as e:
        error_message = e.pgerror
        print("Exception uploading to "+table_name+" table."+error_message)

    finally:
        # Close the cursor and connection
        cursor.close()
        pg_conn.close()


In [ ]:
# def psyco_read_sql(sql):
#     try:
#         pg_conn = psycopg2.connect(
#             dbname=dbname,
#             user=user,
#             password=password,
#             host=host,
#             port=port
#         )
#     except psycopg2.Error as e:
#         error_message = e.pgerror
#         print("Error Connecting:", error_message)

#     # Use pd.read_sql to execute the query and load data into a DataFrame
#     df = pd.read_sql(sql, pg_conn)

#     # Print the DataFrame
#     print(df)

#     # Close the database connection
#     pg_conn.close()


In [ ]:
#function to gather df from json response
def df_from_json(feed):
    response = requests.get(feed)
    json_data = response.json()
    df = pd.DataFrame(json_data)
    return df

# Check for missing players

In [ ]:
#get the players list from datagolf
players_dg = df_from_json('https://feeds.datagolf.com/get-player-list?file_format=json&key='+api_key)

#get the unique dg_id from players already in aws db
players_aws = pd.read_sql(sql = """
                          select distinct dg_id from player;
                          """, con=db_URI)

In [ ]:
#find players that are in the datagolf db that are missing from the aws db
missing_players = players_dg[~players_dg['dg_id'].isin(players_aws['dg_id'])] # ~ operator inverts - this is looking for all golfers in datagolf db that are not in aws
missing_players

In [ ]:
#if there's missing players, add them to the aws db
if(len(missing_players['player_name'])>0):
    insert_df_to_sql('player', missing_players)

# Check for Event Updates

In [ ]:
#dynamic current year so script can be re-used next year
current_year = datetime.now().year

#capture datagolf events and the current year events already uploaded to aws
events_dg = df_from_json('https://feeds.datagolf.com/historical-raw-data/event-list?file_format=json&key='+api_key)
events_aws_current_year = pd.read_sql(sql = f"""
                          select event_id from event where calendar_year = {current_year};
                          """, con=db_URI)

#filter datagolf historical events to current year, pga
events_dg_current_year = events_dg[(events_dg['tour'] == 'pga') & (events_dg['calendar_year'] == events_dg['calendar_year'].max())]

In [ ]:
#find which events are not in aws
missing_events = events_dg_current_year[~events_dg_current_year['event_id'].isin(events_aws_current_year['event_id'])]
missing_events

# Check for DFS event

In [ ]:
#create array to hold missing dfs event ids
no_dfs = []

#loop through each missing event to check for empty response code 400
for id_ev in missing_events['event_id']:
    feed = 'https://feeds.datagolf.com/historical-dfs-data/points?tour=pga&site=fanduel&event_id='+str(id_ev)+'&year='+str(current_year)+'&file_format=json&key='+api_key
    response = requests.get(feed)
    if(response.status_code == 400):
        no_dfs.append(id_ev)

#remove any missing events from missing_events df as the DFS event may not be ready yet (or will never exist)
missing_events = missing_events[~missing_events['event_id'].isin(no_dfs)]
missing_events

In [ ]:
#if there's any missing events, add them to the aws db
if(len(missing_events['event_id'])>0):
    missing_events_w_dfs_payout = missing_events.copy().drop(['sg_categories', 'tour', 'traditional_stats'], axis=1)
    missing_events_w_dfs_payout['dfs_payout'] = None
    insert_df_to_sql('event', missing_events_w_dfs_payout)

In [ ]:
pd.read_sql(f"""select * from event where calendar_year = {current_year} order by date desc""", con=db_URI)

# Parse Raw Player Scoring & Data

In [ ]:
#get headers from an event with everthing documented (event 2 in 2025 is one example)
event_for_names = df_from_json('https://feeds.datagolf.com/historical-raw-data/rounds?tour=pga&event_id=2&year=2025&file_format=json&key='+api_key)
basic_names = ['event_id', 'calendar_year', 'dg_id', 'round', 'fin_text']
stat_names = basic_names + pd.json_normalize(event_for_names['scores'].loc[0]['round_1'], max_level=0).columns.tolist()
stat_names.sort()

#create placeholder array to store round information.  *Getting warnings appending to pre-set dataframe so went the array route for now.  Could potentially pre-define/pre-fill df with None but this gets tricky because some rounds will not be uploaded due to CUT or WD
rounds = []

for id_ev in missing_events['event_id']:
    #get the event historical data
    event_temp = df_from_json('https://feeds.datagolf.com/historical-raw-data/rounds?tour=pga&event_id='+str(id_ev)+'&year='+str(current_year)+'&file_format=json&key='+api_key)
    
    #loop through players
    player_summary = pd.json_normalize(event_temp['scores'], max_level=0)
    for row in player_summary.iterrows():
        player = row[1]

        #create array of round info.  If there is a missing round, do not add
        event_rounds = []
        if pd.notna(player['round_1']):
            event_rounds = event_rounds + [pd.json_normalize(player['round_1'], max_level = 0)]
        if pd.notna(player['round_2']):
            event_rounds = event_rounds + [pd.json_normalize(player['round_2'], max_level = 0)]
        if pd.notna(player['round_3']):
            event_rounds = event_rounds + [pd.json_normalize(player['round_3'], max_level = 0)]
        if pd.notna(player['round_4']):
            event_rounds = event_rounds + [pd.json_normalize(player['round_4'], max_level = 0)]

        for index, rnd in enumerate(event_rounds):
            #add player summary info (basic_names) to each round df and add to rounds
            basic_cols = pd.DataFrame({'dg_id':[player['dg_id']], 'fin_text':[player['fin_text']], 'round':[index+1], 'event_id':[id_ev], 'calendar_year':[current_year]})
            rnd[basic_cols.columns] = basic_cols

            #find differences in the column headers (if any - some events do not contain all the data, so will need to fill with NA)
            diffs = set(stat_names)-set(rnd.columns)

            #add NA to missing columns
            for item in diffs:
                rnd[item] = None

            #sort the columns so array can be easily converted to df later
            rnd_sorted = rnd.copy()[sorted(rnd.columns)]

            #append array
            rounds.append(rnd_sorted.loc[0].values)

#convert to dataframe and normalize NAs
rounds = pd.DataFrame(rounds, columns=stat_names).replace([np.nan, 'missing'], None, inplace=False)
rounds.sample(10)

## Upload Any Missing Courses

In [ ]:
new_courses = rounds.drop_duplicates('course_num')[['course_name', 'course_num', 'course_par']].reset_index().drop('index', axis=1)
courses_aws = pd.read_sql(sql="""SELECT * FROM COURSE;""", con=db_URI)
missing_courses = new_courses[~new_courses['course_num'].isin(courses_aws['course_num'])]
missing_courses

In [ ]:
#if there's missing courses, add them to the aws db
if(len(missing_courses['course_num'])>0):
    insert_df_to_sql('course', missing_courses)

## Upload Player Scoring

In [ ]:
#cleanup rounds df
rounds_clean = rounds.drop(['course_name', 'course_par', 'fin_text'], axis=1)

#get necessary event information
events_aws = pd.read_sql(sql=f"""SELECT id_event, event_id, calendar_year FROM EVENT WHERE CALENDAR_YEAR = {current_year};""", con=db_URI)
events_aws['id_event'] = events_aws['id_event'].astype(str)
events_aws

#get necessary course information
courses_aws = pd.read_sql(sql="""SELECT id_course, course_num FROM COURSE;""", con=db_URI)

#replace aws id's with dg
rounds_clean = pd.merge(rounds_clean, events_aws, on=['event_id', 'calendar_year'], how='left').drop(['event_id', 'calendar_year'], axis=1)
rounds_clean = pd.merge(rounds_clean, courses_aws, on=['course_num'], how='left').drop('course_num', axis=1)

#preview
rounds_clean.sample(10)

In [ ]:
insert_df_to_sql('round', rounds_clean)

# Upload Player DFS

In [ ]:
#get all stat headers for dfs scoring (they all exist for event 2 in 2025)
dfs = df_from_json('https://feeds.datagolf.com/historical-dfs-data/points?tour=pga&site=fanduel&event_id=2&year=2025&file_format=json&key='+api_key)
stat_names = pd.json_normalize(dfs['dfs_points'].loc[0], max_level=0).drop(['ownership', 'player_name'], axis=1).columns.tolist()+['id_event']
stat_names.sort()

#set placeholder array
dfs_array = []

#loop through the missing events
for id_ev in missing_events['event_id']:

    #get the event historical data
    event_temp = df_from_json('https://feeds.datagolf.com/historical-dfs-data/points?tour=pga&site=fanduel&event_id='+str(id_ev)+'&year='+str(current_year)+'&file_format=json&key='+api_key)

    #loop through golfer's dfs event summary
    for player in event_temp['dfs_points']:
        #get the dfs points, add calendar year and event id so it can be merged with the aws id_event
        dfs_pts = pd.json_normalize(player, max_level=0).drop(['player_name', 'ownership'], axis=1)
        dfs_pts['calendar_year'] = current_year
        dfs_pts['event_id'] = id_ev
        dfs_pts = pd.merge(dfs_pts, events_aws, on=['event_id', 'calendar_year']).drop(['calendar_year', 'event_id'], axis=1)
        
        #find differences in the column headers (if any - some events do not contain all the data, so will need to fill with NA)
        diffs = set(stat_names)-set(dfs_pts.columns)

        #add NA to missing columns
        for item in diffs:
            dfs_pts[item] = None
    
        #sort the columns so array can be easily converted to df later
        dfs_pts_sorted = dfs_pts.copy()[sorted(dfs_pts.columns)]

        #append to placeholder array
        dfs_array.append(dfs_pts_sorted.loc[0].values)

#convert to dataframe and normalize NAs
dfs_array = pd.DataFrame(dfs_array, columns=stat_names).replace([np.nan, 'missing'], None, inplace=False)
dfs_array.sample(10)
    

In [ ]:
insert_df_to_sql('dfs_total', dfs_array)